[![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-F37626?logo=jupyter&logoColor=white)](#)
[![Python](https://img.shields.io/pypi/pyversions/oracle-vecdb)](https://pypi.org/project/oracle-vecdb/)
[![oracle-vecdb](https://img.shields.io/badge/oracle-vecdb-2EA44F?logo=oracle&logoColor=white)](#)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/start_here/connect.ipynb)

# Connect to Oracle AI Database with Python

Configure the Oracle VecDB Python SDK to connect to an Oracle AI Database REST endpoint, authenticate as a vector user, and verify the connection. Choose username and password authentication for local development, or OAuth 2.0 client-credentials authentication when your application should use a bearer token.

Run Section 1, then either Section 2 or Section 3, and then Section 4. Section 5 is optional.

## Choose an authentication method

| Authentication method | Use when | Instructions |
| --- | --- | --- |
| Username and password | You are following the quickstart or developing locally. | Configure in Section 2. |
| OAuth 2.0 bearer token | Your application uses service-to-service authentication or should not send the vector user password. | Configure in Section 3. |

The quickstart uses username and password authentication. You can switch to OAuth 2.0 bearer-token authentication without changing the SDK REST endpoint.

## Requirements

- Python 3.10 or later
- Oracle AI Database 23.26.3 or later
- ORDS 26.2.2 or later
- The Oracle VecDB Python SDK installed in this notebook's Python environment

## Before you begin

1. [Choose a database deployment](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/) and complete its setup.
2. Record the vector user name and password, and copy the **SDK REST endpoint**.
3. For OAuth 2.0, create an OAuth client for the vector user and save its client ID and secret.

The database setup pages explain how to create the user, load the `all_MiniLM_L12_v2` model, and obtain the REST endpoint:

- [Autonomous AI Database Serverless](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)
- [Autonomous AI Database Free Container](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)

This notebook uses the SDK REST endpoint, not the Vector Database Console URL.

## 1. Install the SDK

Install the latest version of the SDK in the Python environment used by this notebook. If you use `uv`, run `uv add oracle-vecdb` in your project instead.

In [ ]:
%%bash
python -m pip install --upgrade oracle-vecdb

## 2. Configure username and password authentication

Use the SDK REST endpoint with the vector user credentials created during database setup. Skip this section if you are using OAuth 2.0 bearer-token authentication.

In [ ]:
from oracle_vecdb import Configuration, OracleVecDB

config = Configuration(
    rest_url="https://<host>:<port>/ords/<vector_user>/_/db-api/stable/vecdb/",
    username="<VECTOR_USER>",
    password="<VECTOR_USER_PASSWORD>",
)

vecdb = OracleVecDB(config)

Use the REST URL mapping in the `/ords/<vector_user>/` part of the endpoint. Update `<host>`, `<port>`, and `<vector_user>` with the values from your deployment setup page. Keep the user name and password out of source control.

**Warning:** Use the lowercase REST URL mapping in the `/ords/<vector_user>/` segment. The mapping is separate from the uppercase `<VECTOR_USER>` database user name used for authentication. For example, a database user named `VECTOR_USER` uses `/ords/vector_user/` in the endpoint.

## 3. Configure OAuth 2.0 bearer-token authentication

Use OAuth 2.0 when your application should authenticate with a client ID and secret instead of sending the vector user password. The OAuth client belongs to the vector user and must have the VecDB role.

### 3.1 Create an OAuth client in Database Actions

Sign in to Database Actions as `<VECTOR_USER>`, then complete the following steps:

1. Select **Development**, then select **REST**.
2. Open **Clients** and select **Create OAuth Client**.
3. On the **Client Definition** tab, set **Grant type** to `CLIENT_CRED`.
4. Enter a name and description for the client. You can also provide a support email and, optionally, a support URI.
5. On the **Roles** tab, add `oracle.dbtools.auth.roles.builtin.VecDB` to the client.
6. Select **Create**.

Save the client ID and client secret when they are displayed. The client secret is shown only when the client is created. If you lose it, rotate the secret and update your application.

**Caution:** Treat the client secret like a password. Do not commit it to source control, include it in client-side code, or write it to application logs.

For the general Database Actions OAuth client administration procedure, see [Configure an OAuth Client](https://docs.oracle.com/en/database/oracle/sql-developer-web/sdwad/vdb-configure-oauth-client.html).

### 3.2 Request an access token

Use the client ID and client secret to request an access token from the OAuth token endpoint. Replace `<host>`, `<port>`, and `<vector_user>` before running the next cell. Do not save the client secret in a notebook or source file.

In [ ]:
%%bash
curl -i -k \
  --user "<CLIENT_ID>:<CLIENT_SECRET>" \
  --data "grant_type=client_credentials" \
  "https://<host>:<port>/ords/<vector_user>/oauth/token"

The example includes `-k` so it also works with the self-signed certificate generated by a local ADB-Free container. The option disables TLS certificate verification; use it only for local testing. For an endpoint with a trusted certificate, remove `-k`. For a local container, you can add the certificate to your host trust store by following [Trust the local HTTPS certificate](https://docs.oracle.com/en/database/oracle/26/vecse/). The response contains an access token. Copy the value of `access_token` into the next cell. If the client secret is exposed, rotate it in Database Actions before requesting another token.

### 3.3 Configure the Python client

Pass the access token to `Configuration` instead of the username and password.

In [ ]:
from oracle_vecdb import Configuration, OracleVecDB

config = Configuration(
    rest_url="https://<host>:<port>/ords/<vector_user>/_/db-api/stable/vecdb/",
    access_token="<BEARER_TOKEN>",
)

vecdb = OracleVecDB(config)

## 4. Verify the connection

Call `describe_vector_database()` to confirm that the user can access the VecDB REST API. Run this cell after configuring either username and password or OAuth 2.0 authentication.

In [ ]:
summary = vecdb.describe_vector_database()
print(summary)

The response contains summary counts for the vector tables, vectors, and models in the `<VECTOR_USER>` schema. A successful response confirms that the endpoint and credentials are working. For the method definition, see `describe_vector_database` in the [Oracle AI Database documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/).

## Request a new access token after expiration

The client-credentials flow returns a short-lived access token and an `expires_in` value. When the access token expires, request a new one by repeating the client-credentials request in Section 3.2 with the same client ID and client secret.

Replace the expired bearer token with the new `access_token` value in subsequent SDK requests.

## 5. Configure TLS, a proxy, and retries

Use additional `Configuration` options when your environment requires custom certificate verification, an HTTP proxy, or retries. This section is optional.

In [ ]:
config = Configuration(
    rest_url="https://<host>:<port>/ords/<vector_user>/_/db-api/stable/vecdb/",
    username="<VECTOR_USER>",
    password="<VECTOR_USER_PASSWORD>",
    ssl_ca_cert="/path/to/ca-bundle.pem",
    retries=3,
)

config.proxy = "http://proxy.example.com:8080"
vecdb = OracleVecDB(config)

For all configuration parameters, environment variables, attributes, and methods, see `Configuration` in the [Oracle AI Database documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/).

## Troubleshoot the connection

- **401 Unauthorized**: Check that the credentials belong to `<VECTOR_USER>` and that the password is correct.
- **403 Forbidden**: Check that the user has the privileges required by the deployment setup page and that the REST URL mapping is enabled.
- **Invalid host or endpoint error**: Check that you used the SDK REST endpoint, which contains `/_/db-api/stable/vecdb/`, rather than the Console URL, which contains `/_sdw/?nav=vector-db`.
- **TLS certificate error**: Confirm that the endpoint uses HTTPS and configure `ssl_ca_cert` only when your environment requires a custom CA bundle.

## Related sample

The [Embedding & Ingest Patterns notebook](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/embeddings/embedding-ingest-patterns.ipynb) loads connection settings, creates an `OracleVecDB` client, and reuses it across a complete workflow.

## Next steps

- [Create Vector Tables](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/).
- [Ingest Vector Data](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/).
- [Search and Filter Vectors](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/).
- Read the complete `Configuration` API reference in the [Oracle AI Database documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/).